# 定位与层叠上下文

学习目标：能找出定位参照、诊断粘性定位与遮挡，并区分普通层叠排序和浏览器顶层。

前置知识：正常流、盒模型、overflow 与滚动容器，HTML 按钮与对话框元素。

适用范围：基础定位面向现代浏览器；模态示例需 HTMLDialogElement.showModal() 支持。示例直接使用已声明支持的 API；其余页面不依赖 JavaScript。

工作目录：`content/Web与应用开发/css/`（以下命令从项目根目录切换到这里执行）。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/10-positioning-stacking/。

1. [index.html](scripts/10-positioning-stacking/index.html)：static、relative、absolute、inset 与遮挡。
2. [scroll.html](scripts/10-positioning-stacking/scroll.html)：fixed 参照和 sticky 滚动边界。
3. [stacking.html](scripts/10-positioning-stacking/stacking.html)：父子层叠上下文与 z-index 对照。
4. [top-layer.html](scripts/10-positioning-stacking/top-layer.html)：原生模态对话框与普通层叠内容对照。
5. [styles.css](scripts/10-positioning-stacking/styles.css)：本章定位和层叠样式。
6. [dialog.js](scripts/10-positioning-stacking/dialog.js)：只绑定打开按钮，调用 showModal()；关闭使用原生对话框表单。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/10-positioning-stacking/index.html)。

保存修改后刷新页面。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 本章使用的属性

| 完整属性名 | 中文名称／含义 | 用途或对象 |
| --- | --- | --- |
| position | 定位方式 | 决定偏移方式和是否脱离正常流 |
| top | 上偏移 | 物理上边定位约束 |
| right | 右偏移 | 物理右边定位约束 |
| bottom | 下偏移 | 物理下边定位约束 |
| left | 左偏移 | 物理左边定位约束 |
| inset | 四边偏移简写 | 按上右下左设置物理偏移 |
| z-index | 层叠级别 | 参与本层叠上下文内排序 |
| isolation | 隔离 | isolate 建立独立层叠上下文 |
| transform | 变换 | 本章观察非 none 对包含块的影响 |
| overflow | 溢出处理 | 滚动口与裁剪条件 |
| pointer-events | 指针命中 | 让演示遮盖层不阻止指针点击 |
| background-color | 背景颜色 | 标出重叠区域与模态背景 |

## 2 static 与 relative：移动后是否保留位置

position 的值选择定位方式。static 按正常流排版，物理偏移不用于移动它；relative 先按正常流分配位置，再相对自己的原位置偏移。

top: 16px 对 relative 表示向下偏移 16px，left: 20px 表示向右偏移 20px；负值可朝相反方向移动。它保留原有布局空间，其他盒子不会随视觉移动而重新排版，因此可能发生重叠。

relative 还可在不设置偏移时为绝对定位后代建立参照。不要为了排开整段内容而滥用相对定位，通常应先调整正常流、内外边距或布局方式。

```html
<div class="flow-demo">
  <div class="item static-box">static</div>
  <div class="item following">后续元素</div>
</div>
<div class="flow-demo">
  <div class="item relative-box">relative</div>
  <div class="item following">后续元素</div>
</div>
```

```css
.flow-demo { width: 260px; border: 2px solid #777; margin: 24px 0; }
.item { height: 40px; background: #e0eeee; }
.following { background: #f5dfb2; }
.static-box { position: static; top: 16px; left: 20px; }
.relative-box { position: relative; top: 16px; left: 20px; }
/* static 的偏移不起作用；relative 的盒向右下移动，但后续盒原位不动。
   观察重叠位置，不能把视觉移动当作给后续内容增加了外边距。 */
```

配套文件：[index.html](scripts/10-positioning-stacking/index.html)、[styles.css](scripts/10-positioning-stacking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/10-positioning-stacking/index.html)

## 3 absolute、包含块与 inset

absolute 脱离正常流，不给兄弟元素预留空间。包含块（containing block）是尺寸和定位计算的参照矩形，不等于直接父元素。

对于本例的普通块祖先，绝对定位沿祖先链寻找最近建立绝对定位包含块的元素；position 非 static 的祖先会建立这种参照，常用 position: relative。其内边距边界作为本例参照。没有合适祖先时参照初始包含块；变换等属性也可能建立包含块，下一节继续比较。

top/bottom 的百分比参照包含块高度，left/right 的百分比参照宽度。inset 是物理四边简写，四值顺序上、右、下、左；单值作用于四边，不随书写方向改变。

对普通非替换绝对定位盒，若同一轴两侧偏移确定且尺寸为 auto，通常会填充剩余空间。若宽高、两侧偏移和边距一起约束，需按定位算法解决，不能把所有设定都当作独立保证。auto 也不总等于 0，可能涉及元素未定位时的静态位置。

```html
<div class="positioned-card">
  <div class="unpositioned-wrapper"><span class="badge">角标</span></div>
  <p>卡片正文仍从正常流起点排版。</p>
</div>
<div class="positioned-card">
  <div class="inset-fill">四边缩进形成的内部区域</div>
</div>
```

```css
.positioned-card { position: relative; width: 280px; height: 120px; padding: 16px; border: 4px solid #777; margin: 24px 0; }
.badge { position: absolute; top: 0; right: 0; width: 80px; height: 28px; background: #f5dfb2; }
.inset-fill { position: absolute; inset: 12px; border: 2px dashed #006666; }
/* 角标参照 positioned-card 的内边距边界，不是直接父 div。
   inset-fill 未设宽高，用四边偏移填充剩余区域；改卡片尺寸再检查。 */
```

配套文件：[index.html](scripts/10-positioning-stacking/index.html)、[styles.css](scripts/10-positioning-stacking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/10-positioning-stacking/index.html)

## 4 fixed 通常参照视口，但有例外

fixed 同样脱离正常流。连续屏幕媒体中，没有建立固定定位包含块的祖先时，它通常相对视口定位，页面滚动不会改变该参照；打印媒体则涉及页面区域。

祖先存在非 none 的 transform 等属性时，可能改由祖先建立固定定位包含块。本例 translate(0) 是不产生视觉位移的变换，却仍满足这个条件。单独给祖先 position: relative 并不把 fixed 改为相对该祖先。

“包含块”解释位置和百分比，“层叠上下文”解释重叠顺序；它们是两种机制，不能看到一个新层叠上下文就推断所有定位参照都变了。

```html
<p class="fixed-note">视口底部固定提示</p>
<div class="transformed">
  <p class="local-fixed">变换祖先内的 fixed</p>
  <p>滚动页面，比较这条提示和视口底部提示。</p>
</div>
```

```css
.fixed-note { position: fixed; left: 12px; right: 12px; bottom: 8px; margin: 0; padding: 8px; background: #e0eeee; z-index: 5; }
.transformed { transform: translate(0); height: 180px; border: 4px solid #777; padding: 40px 16px 16px; }
.local-fixed { position: fixed; top: 0; right: 0; margin: 0; background: #f5dfb2; }
/* translate(0) 视觉上不移动，却仍是非 none 变换：它建立 fixed 包含块。
   页面滚动时 local-fixed 随祖先走，fixed-note 仍相对视口底部定位。 */
```

配套文件：[scroll.html](scripts/10-positioning-stacking/scroll.html)、[styles.css](scripts/10-positioning-stacking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/10-positioning-stacking/scroll.html)

## 5 sticky 的阈值、正常流与容器边界

sticky 始终参与正常流并保留原位置。滚动时，浏览器可按最近滚动口（scrollport）的偏移阈值调整它的视觉位置；它不会在到达阈值后变成脱离正常流的 fixed。

top: 0 约束本例的上边：原位置滚到滚动口顶部附近后，标题向下补偿滚动，直到所属包含块的边界迫使它离开。滚动口负责阈值，包含块限制移动范围，二者职责不同。

粘性定位没有预期效果时依次检查：

- 目标轴是否有非 auto 的偏移，例如 top；两侧都是 auto 时该轴没有粘性约束。
- 最近具有滚动机制的祖先是否就是正在滚动的容器。overflow: hidden 等也可能改变这个祖先，实际没滚动的祖先常令人误判。
- 容器是否有足够高度差和滚动空间；项目被拉伸到容器高度、父容器过短或标题过高，都可能让可移动范围很小。
- 是否被其他内容覆盖，或与另一个 sticky 标题重叠。sticky 的有效定位不等于它一定处于最上方。

下例容器自身可滚动，便于把页面滚动与容器滚动分开观察。

```html
<div class="scroll-box good" tabindex="0" role="region" aria-label="有 top 的粘性定位">
  <div class="sticky-chapter">
    <div class="before">滚动到标题</div>
    <h3 class="sticky-label">章节标题</h3>
    <div class="after">章节正文：继续滚动，观察标题何时离开。</div>
  </div>
  <div class="tail">下一块内容</div>
</div>
<div class="scroll-box no-inset" tabindex="0" role="region" aria-label="缺少 top 的对照">
  <div class="sticky-chapter">
    <div class="before">滚动到标题</div>
    <h3 class="sticky-label">章节标题</h3>
    <div class="after">章节正文：继续滚动，观察标题何时离开。</div>
  </div>
  <div class="tail">下一块内容</div>
</div>
```

```css
.scroll-box { width: 320px; max-width: 100%; height: 220px; overflow: auto; border: 2px solid #777; margin: 20px 0; }
.sticky-chapter { height: 400px; }
.before { height: 60px; }
.sticky-label { position: sticky; top: 0; height: 40px; margin: 0; background: #e0eeee; }
.after { height: 300px; }
.tail { height: 200px; background: #f5dfb2; }
.no-inset .sticky-label { top: auto; }
/* good 标题到滚动口顶部后暂时停留；所属章节离开时，标题一起离开。
   no-inset 在纵轴没有非 auto 偏移，标题按正常位置滚走。
   sticky 始终在正常流中保留位置，后面的正文不会补占标题原位置。 */
```

配套文件：[scroll.html](scripts/10-positioning-stacking/scroll.html)、[styles.css](scripts/10-positioning-stacking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/10-positioning-stacking/scroll.html)

## 6 z-index 只能在相应层叠上下文里比较

判断遮挡时，先找两个元素分别属于哪个层叠上下文（stacking context）。每个上下文作为一个整体参与外层排序；内部很大的 z-index 只能解决内部比较。

z-index 对定位元素有效，也能用于 Flex/Grid 项目而不必设置非 static 定位。relative/absolute 配合非 auto 的 z-index 会创建上下文；0 是整数层级，与 auto 不等价。fixed、sticky 本身会创建上下文。

常见其他触发条件包括 opacity 小于 1、非 none 的 transform 和 isolation: isolate。这里只列与本章相关条件，完整列表见篇末来源。不是每个 DOM 元素都自动形成一个上下文。

同一上下文内也不是只按 DOM 顺序绘制：背景、负层级、正常流和定位内容有各自阶段；整数层级比较后，相同层级通常再结合绘制或文档顺序。排错先找双方所属的上下文，再修改应当参与外层比较的元素，不要不断放大子元素数字。

![外层比较 A 的1与 B 的2；9999位于 A 内部，不能与 B 的2直接比较。](image/illustration/10-01-stacking-context-tree.svg)

图 1：依据 CSS 的层叠上下文原理自行绘制，署名 CMYK Labs，表示下方遮挡实验的比较层级。A 的子元素虽然写 9999，仍随 A 整体处于 B 的下方；图不展开同一上下文内的完整绘制阶段。

运行示例后先修改子元素数字，再把 A 的 1 改成 3，比较两种操作中哪一种真正改变了外层关系。

```html
<div class="stack-stage">
  <div class="context-a">上下文 A：1<div class="high-child">A 的子元素：9999</div></div>
  <div class="context-b">上下文 B：2</div>
</div>
```

```css
.stack-stage { position: relative; isolation: isolate; width: 360px; max-width: 100%; height: 200px; }
.context-a { position: absolute; left: 0; top: 0; width: 220px; height: 150px; z-index: 1; background: #e0eeee; }
.high-child { position: absolute; left: 120px; top: 50px; width: 180px; height: 80px; z-index: 9999; background: #bedddd; }
.context-b { position: absolute; left: 160px; top: 80px; width: 180px; height: 100px; z-index: 2; background: #f5dfb2; }
/* B 遮住 A 的子元素，9999 只在 A 内比较；把 A 的 z-index 改成 3 再看。
   isolation: isolate 为本实验建立独立层叠上下文，不改变其正常流位置。 */
```

配套文件：[stacking.html](scripts/10-positioning-stacking/stacking.html)、[styles.css](scripts/10-positioning-stacking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/10-positioning-stacking/stacking.html)

## 7 顶层不是更大的 z-index

顶层（top layer）由浏览器管理，绘制在普通文档内容上方。成功用 showModal() 打开的 &lt;dialog&gt;、显示中的 popover、全屏元素等可以进入顶层；写一个更大的 z-index 不能让普通元素进入顶层。

只给 &lt;dialog&gt; 加 open 或调用 show()，不等于 showModal() 的模态顶层行为。进入顶层的元素仍保留原来的 DOM 关系，但普通祖先的层叠、变换和裁剪不能按原方式把它压在下方；顶层内顺序也不靠普通 z-index 竞争。

::backdrop 是对应背景伪元素。模态对话框还会让其余文档处于不可交互状态，这是 HTML 对话框行为，不是 CSS 遮罩自动实现的。焦点、关闭和 Escape 行为都需要检查。

本例 [dialog.js](scripts/10-positioning-stacking/dialog.js) 直接绑定打开按钮并调用 showModal()；关闭采用 method="dialog" 的原生表单。CSS Position 4 的顶层表述仍随规范演进，具体 API 支持以目标浏览器为准。

```html
<div class="clipped-container">
  <button id="open-dialog" type="button">打开模态提示</button>
  <dialog id="reading-dialog" aria-labelledby="dialog-title">
    <h2 id="dialog-title">阅读提示</h2>
    <p>每次练习后保存源文件，再刷新页面检查。</p>
    <form method="dialog"><button autofocus>关闭</button></form>
  </dialog>
</div>
<div class="ordinary-cover">普通内容：z-index 为 2147483647</div>
<script src="dialog.js"></script>
```

```javascript
const opener = document.querySelector("#open-dialog");
const dialog = document.querySelector("#reading-dialog");
opener.addEventListener("click", () => dialog.showModal());
// 点击后显示模态并聚焦关闭按钮；Escape 或原生表单关闭。
```

```css
.clipped-container { height: 100px; overflow: hidden; transform: translate(0); border: 2px solid #777; }
.ordinary-cover { position: fixed; top: 45%; left: 10%; right: 10%; height: 160px; z-index: 2147483647; background: #f5dfb2; pointer-events: none; }
dialog { z-index: -1; width: 26rem; max-width: calc(100% - 4rem); border: 2px solid #006666; padding: 20px; }
dialog:not([open]) { display: none; }
dialog::backdrop { background-color: rgb(0 0 0 / 25%); }
/* showModal() 成功后，对话框进入顶层；即使 z-index 为 -1 仍在普通内容之上。
   它不再被原祖先的 overflow 裁剪；按 Escape 或关闭按钮退出。 */
```

配套文件：[top-layer.html](scripts/10-positioning-stacking/top-layer.html)、[styles.css](scripts/10-positioning-stacking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/10-positioning-stacking/top-layer.html)

## 8 诊断遮挡时同时看布局和绘制

先确认元素是否仍在正常流、参照哪个包含块，再查计算后的偏移、尺寸、滚动祖先和层叠上下文。声明有效但参照不合预期，与语法无效是不同问题。

absolute/fixed 不预留正常流空间，若必须覆盖内容，就为实际覆盖范围安排空间；文字增加、语言改变和浏览器缩放后重新检查。没有稳定覆盖需求时，让内容回到正常流通常更可靠。

提高 z-index 不能解决父容器裁剪，也不能把必要正文从另一块覆盖内容中自动挤开。键盘焦点落到被遮挡控件时，同样属于需要处理的问题。

```html
<div class="obstruction">
  <p class="overlay-label">固定在容器角落的提示</p>
  <p>放大字体后，仍须检查这段内容是否被提示遮住。</p>
</div>
```

```css
.obstruction { position: relative; max-width: 30rem; padding-top: 4rem; border: 1px solid #777; }
.overlay-label { position: absolute; top: 0; left: 0; margin: 0; padding: 8px; background: #e0eeee; }
/* 4rem 是本例给提示预留的空间，并非所有字号和文本长度都安全。
   增加提示文字或放大字体后重新核对；必要时让提示回到正常流。 */
```

配套文件：[index.html](scripts/10-positioning-stacking/index.html)、[styles.css](scripts/10-positioning-stacking/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/10-positioning-stacking/index.html)

## 本章小结

- relative 和 sticky 保留正常流空间；absolute 与 fixed 脱离正常流。
- 包含块负责定位参照；变换祖先可以改变 fixed 的参照。
- sticky 需要偏移阈值和实际移动空间，且受所属包含块限制。
- z-index 在层叠上下文中比较，子元素无法独自跨越父上下文。
- 顶层由浏览器与相关 API 管理，模态行为不等于画一个高层级遮罩。

## 练习

（1）去掉卡片的 position: relative，预测角标改用哪个参照，并在开发者工具中确认；随后恢复。

（2）滚动 good 容器到标题粘住，再滚动到底。检查标题是否随所属章节离开，以及正文是否始终保留标题的原有布局空间。

（3）只把 A 子元素的 z-index 加大，检查能否盖住 B；再修改 A 自身。最后打开模态对话框，比较这些普通上下文与顶层的关系。

### 提示

第一题从完整祖先链找包含块；第三题先画出 A、B 的父子上下文关系，不能只比较页面中最大的整数。

### 重点题解析

第（1）题中其他祖先未设置定位、变换或包含属性，去掉卡片的 relative 后，角标参照初始包含块，出现在页面右上方；它仍是 absolute，不会因此变成 fixed。

第（3）题保持 A 的层级 1、B 的层级 2 时，提高子元素的 9999 不会改变外层排序；把 A 改为 3 才让 A 整体位于 B 上面。showModal() 使用浏览器顶层，普通文档再高的 z-index 仍不能压到模态之上。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| W3C | [CSS 2.2 附录 E](https://www.w3.org/TR/CSS22/zindex.html) 的层叠上下文与绘制顺序；[CSS Positioned Layout 3 §2](https://www.w3.org/TR/css-position-3/#position-property) 的定位类型和包含块，[§3](https://www.w3.org/TR/css-position-3/#insets) 的偏移，[§3.4](https://www.w3.org/TR/css-position-3/#sticky-pos) 的粘性视图矩形与包含块限制；[CSS Positioned Layout 4 §3 Top Layer](https://www.w3.org/TR/css-position-4/#top-layer) 的顶层绘制、层叠和祖先影响；[CSS Flexbox 1 §4.3](https://www.w3.org/TR/css-flexbox-1/#painting) 的 Flex 项目绘制与 z-index。 |
| MDN | [position](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/position) 的正常流、偏移、fixed 和 sticky 条件；[Containing block](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Display/Containing_block) 的定位参照及变换祖先；[Stacking context](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Positioned_layout/Stacking_context) 的触发条件和父子上下文；[Top layer](https://developer.mozilla.org/en-US/docs/Glossary/Top_layer)、[HTMLDialogElement.showModal()](https://developer.mozilla.org/en-US/docs/Web/API/HTMLDialogElement/showModal) 的顶层、背景与模态行为及兼容资料。 |
| WHATWG HTML | [The dialog element](https://html.spec.whatwg.org/multipage/interactive-elements.html#the-dialog-element) 的模态显示、焦点、关闭及 method="dialog" 表单。 |
| Python 3.12 | [http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface)。 |